In [ ]:
## Author  : Manikandan J
## Dataset : Luxury_Housing_Bangalore.csv (101,000 rows)
## Output  : luxury_housing_cleaned.csv

In [35]:
import pandas as pd
import numpy as np
import re
import os

In [36]:
INPUT_PATH  = "Luxury_Housing_Bangalore.csv"
OUTPUT_PATH = "luxury_housing_cleaned.csv"

In [ ]:
# 1. LOAD DATA

def load_data(path: str) -> pd.DataFrame:
    """Load raw CSV into a DataFrame."""
    df = pd.read_csv(path)
    print(f"[LOAD] Rows: {len(df):,} | Columns: {df.shape[1]}")
    return df

In [ ]:
# 2. REMOVE DUPLICATES

def remove_duplicates(df: pd.DataFrame) -> pd.DataFrame:
    """Drop fully duplicate rows."""
    before = len(df)
    df = df.drop_duplicates()
    dropped = before - len(df)
    print(f"[DUPLICATES] Removed {dropped:,} duplicate rows → {len(df):,} rows remain")
    return df.reset_index(drop=True)

In [39]:
# 3. CLEAN Ticket_Price_Cr
#    Mixed formats: plain float + "₹9.82 Cr" strings

def clean_ticket_price(df: pd.DataFrame) -> pd.DataFrame:
    """
    Strip currency symbols and 'Cr' text, then convert to float.
    Rows with unparseable values → NaN (handled in null treatment).
    """
    def parse_price(val):
        if pd.isna(val):
            return np.nan
        val = str(val).strip()
        # Remove ₹, 'Cr', whitespace
        val = re.sub(r'[₹\s]', '', val)
        val = re.sub(r'(?i)cr$', '', val)
        try:
            return float(val)
        except ValueError:
            return np.nan
 
    df['Ticket_Price_Cr'] = df['Ticket_Price_Cr'].apply(parse_price)
    print(f"[PRICE] Ticket_Price_Cr converted to float. Nulls: {df['Ticket_Price_Cr'].isna().sum():,}")
    return df

In [40]:
# 4. NORMALIZE TEXT FIELDS

def remove_duplicates(df: pd.DataFrame) -> pd.DataFrame:
    """Drop fully duplicate rows."""
    before = len(df)
    df = df.drop_duplicates()
    dropped = before - len(df)
    print(f"[DUPLICATES] Removed {dropped:,} duplicate rows → {len(df):,} rows remain")
    return df.reset_index(drop=True)
 
 
# ─────────────────────────────────────────────
# 3. CLEAN Ticket_Price_Cr
#    Mixed formats: plain float + "₹9.82 Cr" strings
# ─────────────────────────────────────────────
def clean_ticket_price(df: pd.DataFrame) -> pd.DataFrame:
    """
    Strip currency symbols and 'Cr' text, then convert to float.
    Rows with unparseable values → NaN (handled in null treatment).
    """
    def parse_price(val):
        if pd.isna(val):
            return np.nan
        val = str(val).strip()
        # Remove ₹, 'Cr', whitespace
        val = re.sub(r'[₹\s]', '', val)
        val = re.sub(r'(?i)cr$', '', val)
        try:
            return float(val)
        except ValueError:
            return np.nan
 
    df['Ticket_Price_Cr'] = df['Ticket_Price_Cr'].apply(parse_price)
    print(f"[PRICE] Ticket_Price_Cr converted to float. Nulls: {df['Ticket_Price_Cr'].isna().sum():,}")
    return df
 
 
# ─────────────────────────────────────────────
# 4. NORMALIZE TEXT FIELDS
#    Configuration  → uppercase standard  (3BHK / 4BHK / 5BHK+)
#    Micro_Market   → Title Case
#    NRI_Buyer      → 1 / 0
# ─────────────────────────────────────────────
def normalize_text_fields(df: pd.DataFrame) -> pd.DataFrame:
    # Configuration: strip spaces, uppercase, standardize
    df['Configuration'] = (
        df['Configuration']
        .str.strip()
        .str.upper()
        .str.replace(r'\s+', '', regex=True)
    )
    print(f"[CONFIG] Unique configs after normalize: {sorted(df['Configuration'].unique())}")
 
    # Micro_Market: strip + Title Case
    df['Micro_Market'] = (
        df['Micro_Market']
        .str.strip()
        .str.title()
    )
    print(f"[MARKET] Unique Micro_Markets after normalize: {df['Micro_Market'].nunique()}")
 
    # NRI_Buyer: yes→1, no→0
    df['NRI_Buyer'] = df['NRI_Buyer'].str.strip().str.lower().map({'yes': 1, 'no': 0})
    print(f"[NRI] NRI_Buyer mapped to 0/1. Nulls: {df['NRI_Buyer'].isna().sum()}")
 
    # Standardize other string columns (strip whitespace)
    for col in ['Transaction_Type', 'Buyer_Type', 'Possession_Status', 'Sales_Channel', 'Developer_Name']:
        df[col] = df[col].str.strip()
 
    return df

In [41]:
# 5. HANDLE NULLS

def handle_nulls(df: pd.DataFrame) -> pd.DataFrame:
    """
    - Unit_Size_Sqft  : fill nulls with median; remove rows where value ≤ 0
    - Ticket_Price_Cr : fill nulls with median
    - Amenity_Score   : fill nulls with mean (score field)
    - Buyer_Comments  : fill nulls with 'No Comment'
    """
    # Fix negative / zero Unit_Size_Sqft before median calc
    invalid_size = df['Unit_Size_Sqft'] <= 0
    print(f"[SQFT] Replacing {invalid_size.sum()} invalid (≤0) Unit_Size_Sqft values with NaN")
    df.loc[invalid_size, 'Unit_Size_Sqft'] = np.nan
 
    median_sqft = df['Unit_Size_Sqft'].median()
    df['Unit_Size_Sqft'] = df['Unit_Size_Sqft'].fillna(median_sqft).round(2)
    print(f"[SQFT] Nulls filled with median = {median_sqft:.2f}")
 
    median_price = df['Ticket_Price_Cr'].median()
    df['Ticket_Price_Cr'] = df['Ticket_Price_Cr'].fillna(median_price).round(4)
    print(f"[PRICE] Nulls filled with median = {median_price:.4f}")
 
    mean_amenity = df['Amenity_Score'].mean()
    df['Amenity_Score'] = df['Amenity_Score'].fillna(mean_amenity).round(4)
    print(f"[AMENITY] Nulls filled with mean = {mean_amenity:.4f}")
 
    df['Buyer_Comments'] = df['Buyer_Comments'].fillna('No Comment')
    print(f"[COMMENTS] Nulls filled with 'No Comment'")
 
    return df

In [42]:
# 6. FEATURE ENGINEERING

def feature_engineering(df: pd.DataFrame) -> pd.DataFrame:
    """
    Derived columns:
    - Price_per_Sqft    : Ticket_Price_Cr (in lakhs) / Unit_Size_Sqft
    - Quarter_Number    : fiscal quarter label  (Q1 2024, Q2 2024 …)
    - Booking_Flag      : 1 if Possession_Status != 'Launch', else 0
    - Price_Segment     : Budget / Mid / Premium / Ultra-Premium
    - Connectivity_Band : Low / Medium / High  (based on Connectivity_Score)
    """
 
    # 6a. Price per Sqft  (1 Cr = 100 Lakhs → price_lakhs / sqft)
    price_lakhs = df['Ticket_Price_Cr'] * 100          # convert Cr to Lakhs
    df['Price_per_Sqft'] = (price_lakhs / df['Unit_Size_Sqft']).round(2)
    print(f"[FEAT] Price_per_Sqft created. Sample mean: {df['Price_per_Sqft'].mean():.2f}")
 
    # 6b. Quarter Number from Purchase_Quarter (date string like "2024-06-30")
    df['Purchase_Quarter'] = pd.to_datetime(df['Purchase_Quarter'], errors='coerce')
    df['Quarter_Number'] = df['Purchase_Quarter'].dt.to_period('Q').astype(str)
    print(f"[FEAT] Quarter_Number created. Unique: {sorted(df['Quarter_Number'].unique())}")
 
    # 6c. Booking Flag
    # Treat 'Ready to move' and 'Under construction' as booked; 'Launch' as not yet booked
    df['Booking_Flag'] = df['Possession_Status'].apply(
        lambda x: 0 if str(x).strip().lower() == 'launch' else 1
    )
    print(f"[FEAT] Booking_Flag created. Booked: {df['Booking_Flag'].sum():,} / Not booked: {(df['Booking_Flag']==0).sum():,}")
 
    # 6d. Price Segment
    def segment_price(price):
        if price < 5:
            return 'Budget'
        elif price < 10:
            return 'Mid'
        elif price < 15:
            return 'Premium'
        else:
            return 'Ultra-Premium'
 
    df['Price_Segment'] = df['Ticket_Price_Cr'].apply(segment_price)
    print(f"[FEAT] Price_Segment distribution:\n{df['Price_Segment'].value_counts().to_string()}")
 
    # 6e. Connectivity Band
    def connectivity_band(score):
        if score < 4:
            return 'Low'
        elif score < 7:
            return 'Medium'
        else:
            return 'High'
 
    df['Connectivity_Band'] = df['Connectivity_Score'].apply(connectivity_band)
    print(f"[FEAT] Connectivity_Band distribution:\n{df['Connectivity_Band'].value_counts().to_string()}")
 
    return df

In [43]:
# 7. FINAL VALIDATION

def validate(df: pd.DataFrame) -> None:
    """Print a final summary of the cleaned DataFrame."""
    print("\n" + "="*50)
    print("FINAL DATASET SUMMARY")
    print("="*50)
    print(f"Rows    : {len(df):,}")
    print(f"Columns : {df.shape[1]}")
    print(f"\nNull counts (should all be 0 except Buyer_Comments handled):")
    print(df.isnull().sum()[df.isnull().sum() > 0])
    print(f"\nDtype overview:")
    print(df.dtypes)
    print(f"\nSample (first 3 rows):")
    print(df.head(3).to_string())

In [44]:
# 8. SAVE OUTPUT

def save_data(df: pd.DataFrame, path: str) -> None:
    df.to_csv(path, index=False)
    print(f"\n[SAVE] Cleaned data saved → {path}  ({os.path.getsize(path)/1024/1024:.1f} MB)")

In [45]:
# MAIN PIPELINE

def run_pipeline(input_path: str = INPUT_PATH, output_path: str = OUTPUT_PATH) -> pd.DataFrame:
    print("\n" + "="*50)
    print("STARTING DATA CLEANING PIPELINE")
    print("="*50 + "\n")
 
    df = load_data(input_path)
    df = remove_duplicates(df)
    df = clean_ticket_price(df)
    df = normalize_text_fields(df)
    df = handle_nulls(df)
    df = feature_engineering(df)
    validate(df)
    save_data(df, output_path)
 
    print("\n✅ Pipeline complete!")
    return df
 
 
if __name__ == "__main__":
    df_clean = run_pipeline(
        input_path=INPUT_PATH,
        output_path=OUTPUT_PATH
    )


STARTING DATA CLEANING PIPELINE

[LOAD] Rows: 101,000 | Columns: 18
[DUPLICATES] Removed 1,000 duplicate rows → 100,000 rows remain
[PRICE] Ticket_Price_Cr converted to float. Nulls: 9,913
[CONFIG] Unique configs after normalize: ['3BHK', '4BHK', '5BHK+']
[MARKET] Unique Micro_Markets after normalize: 16
[NRI] NRI_Buyer mapped to 0/1. Nulls: 0
[SQFT] Replacing 500 invalid (≤0) Unit_Size_Sqft values with NaN
[SQFT] Nulls filled with median = 6004.00
[PRICE] Nulls filled with median = 12.0385
[AMENITY] Nulls filled with mean = 7.5042
[COMMENTS] Nulls filled with 'No Comment'
[FEAT] Price_per_Sqft created. Sample mean: 0.23
[FEAT] Quarter_Number created. Unique: ['2023Q2', '2023Q3', '2023Q4', '2024Q1', '2024Q2', '2024Q3', '2024Q4', '2025Q1']
[FEAT] Booking_Flag created. Booked: 66,659 / Not booked: 33,341
[FEAT] Price_Segment distribution:
Price_Segment
Premium          62570
Mid              21492
Ultra-Premium    15045
Budget             893
[FEAT] Connectivity_Band distribution:
Conne

In [56]:
import pandas as pd
from sqlalchemy import create_engine, text

# ── CONFIG ──────────────────────────────────
DB_USER  = "root"
DB_PASS  = "1234567890"
DB_HOST  = "localhost"
DB_PORT  = 3306
DB_NAME  = "luxury_housing_db"
CLEANED_CSV = "luxury_housing_cleaned.csv"
TABLE_NAME  = "luxury_housing"
CHUNK_SIZE  = 5000

# ── CONNECT ─────────────────────────────────
conn_str = f"mysql+pymysql://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}?charset=utf8mb4"
engine = create_engine(conn_str, echo=False)
print("✅ Connected!")

# ── LOAD CSV ─────────────────────────────────
df = pd.read_csv(CLEANED_CSV, parse_dates=['Purchase_Quarter'])
df.columns = [c.lower() for c in df.columns]
df['nri_buyer']        = df['nri_buyer'].astype(int)
df['booking_flag']     = df['booking_flag'].astype(int)
df['buyer_comments']   = df['buyer_comments'].str[:500]
df['purchase_quarter'] = df['purchase_quarter'].astype(str)
df['quarter_number']   = df['quarter_number'].astype(str)
print(f"✅ CSV loaded — {len(df):,} rows")

# ── WIPE TABLE & RELOAD ALL ROWS ─────────────
total   = len(df)
batches = (total // CHUNK_SIZE) + 1

for i, start in enumerate(range(0, total, CHUNK_SIZE)):
    chunk     = df.iloc[start : start + CHUNK_SIZE]
    mode      = 'replace' if i == 0 else 'append'   # replace only on first batch
    chunk.to_sql(TABLE_NAME, con=engine, if_exists=mode, index=False, method='multi')
    pct       = min(100, round((start + len(chunk)) / total * 100))
    print(f"  Batch {i+1:>2}/{batches} — {start + len(chunk):>7,} / {total:,} rows ({pct}%)")

print("\n✅ All 100,000 rows loaded!")

# ── VALIDATE ─────────────────────────────────
with engine.connect() as con:
    result = pd.read_sql(text("SELECT COUNT(*) AS total_rows FROM luxury_housing"), con)
    print(result)

✅ Connected!
✅ CSV loaded — 100,000 rows
  Batch  1/21 —   5,000 / 100,000 rows (5%)
  Batch  2/21 —  10,000 / 100,000 rows (10%)
  Batch  3/21 —  15,000 / 100,000 rows (15%)
  Batch  4/21 —  20,000 / 100,000 rows (20%)
  Batch  5/21 —  25,000 / 100,000 rows (25%)
  Batch  6/21 —  30,000 / 100,000 rows (30%)
  Batch  7/21 —  35,000 / 100,000 rows (35%)
  Batch  8/21 —  40,000 / 100,000 rows (40%)
  Batch  9/21 —  45,000 / 100,000 rows (45%)
  Batch 10/21 —  50,000 / 100,000 rows (50%)
  Batch 11/21 —  55,000 / 100,000 rows (55%)
  Batch 12/21 —  60,000 / 100,000 rows (60%)
  Batch 13/21 —  65,000 / 100,000 rows (65%)
  Batch 14/21 —  70,000 / 100,000 rows (70%)
  Batch 15/21 —  75,000 / 100,000 rows (75%)
  Batch 16/21 —  80,000 / 100,000 rows (80%)
  Batch 17/21 —  85,000 / 100,000 rows (85%)
  Batch 18/21 —  90,000 / 100,000 rows (90%)
  Batch 19/21 —  95,000 / 100,000 rows (95%)
  Batch 20/21 — 100,000 / 100,000 rows (100%)

✅ All 100,000 rows loaded!
   total_rows
0      100000
